# 01 — Analisi del Portfolio (Fase 1)

Questo notebook mostra l'intero flusso della **Fase 1** di `trading-ai`:

1. Caricamento dei dati (qui usiamo dati di esempio che imitano il formato reale dei broker)
2. Calcolo di tutte le metriche di portfolio (P&L FIFO, rendimenti, rischio, allocazione)
3. Visualizzazione interattiva con Plotly

> **Nota:** i prezzi sono *iniettati* da uno storico sintetico per garantire riproducibilità offline.
> Con i CSV reali in `data/raw/`, basta usare `loader.load_transactions()` e rimuovere gli override `price_data` / `current_prices` per scaricare i prezzi veri da yfinance.

## 0. Setup
Aggiungiamo la root del progetto al `sys.path` così possiamo importare il package `src` senza installarlo, e abilitiamo il rendering Plotly inline.

In [ ]:
import sys
from pathlib import Path

# Root = cartella che contiene `src/` (un livello sopra `notebooks/`).
ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = 'notebook'

import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Root progetto:', ROOT)

## 1. Caricamento dati
Generiamo transazioni di esempio nello **schema canonico** (`date, ticker, type, quantity, price, amount_eur, currency, fx_rate, broker`).
Il dataset include acquisti multi-lotto, una vendita parziale, una posizione in USD, dividendi, commissioni e imposte.

Con dati reali sostituire con:
```python
from src.ingestion import loader
tx = loader.load_transactions()      # legge tutti i CSV in data/raw/
loader.save_processed(tx)            # salva in data/processed/transactions.parquet
```

In [ ]:
from src.ingestion.sample_data import make_sample_transactions, make_sample_price_history

tx = make_sample_transactions()
price_history = make_sample_price_history(tx)
# Prezzi correnti fittizi (EUR) per l'unrealized; con dati reali ometterli per usare yfinance.
current_prices = {'VWCE.DE': 130.0, 'AAPL': 200.0, 'ENI.MI': 15.0}

tx

## 2. Metriche di portfolio

### 2.1 Posizioni attuali e P&L realizzato (FIFO)
Il P&L realizzato abbina le vendite ai lotti di acquisto più vecchi (FIFO), gestendo le vendite parziali.

In [ ]:
from src.analysis import metrics

print('Posizioni attuali (quantità):')
display(metrics.current_holdings(tx).to_frame())

print('\nP&L realizzato (FIFO) per ticker:')
realized = metrics.realized_pnl(tx)
display(realized)

### 2.2 P&L non realizzato (mark-to-market)
Confronta il costo dei lotti residui col valore di mercato attuale.

In [ ]:
unrealized = metrics.unrealized_pnl(tx, current_prices=current_prices)
display(unrealized)

### 2.3 Valore del portfolio nel tempo
Posizioni cumulate giornaliere valorizzate con lo storico prezzi, più la componente di cassa.

In [ ]:
pv = metrics.portfolio_value_over_time(tx, price_data=price_history)
pv.tail()

### 2.4 Rendimenti (TWR / MWR) e metriche di rischio

In [ ]:
ret = metrics.total_return(tx, price_data=price_history, current_prices=current_prices)
# Le metriche di rischio usano holdings_value (valore di mercato delle posizioni):
# total_value include la cassa che, senza i versamenti tracciati, puo' essere negativa.
mdd = metrics.max_drawdown(pv['holdings_value'])
sharpe = metrics.sharpe_ratio(pv['holdings_value'])

print(f"TWR (time-weighted):  {ret['twr']:.2%}")
print(f"MWR / XIRR (annuo):   {ret['mwr']:.2%}")
print(f"Valore finale:        EUR {ret['final_value']:,.2f}")
print(f"Max Drawdown:         {mdd:.2%}")
print(f"Sharpe ratio:         {sharpe:.2f}")

### 2.5 Allocazione attuale

In [ ]:
alloc = metrics.allocation_breakdown(tx, current_prices=current_prices)
display(alloc)
print('\nPer asset class:')
display(alloc.groupby('asset_class')['weight_pct'].sum().sort_values(ascending=False).to_frame())

## 3. Visualizzazioni (Plotly)

### 3.1 Valore del portfolio con shading del drawdown

In [ ]:
from src.analysis import charts

charts.plot_portfolio_value(pv).show()

### 3.2 Allocazione (torta interattiva)

In [ ]:
charts.plot_allocation_pie(alloc).show()

### 3.3 Waterfall del P&L realizzato

In [ ]:
charts.plot_pnl_waterfall(realized).show()

### 3.4 Heatmap dei rendimenti mensili

In [ ]:
charts.plot_monthly_returns_heatmap(pv).show()

---
## Prossimi passi
- **Fase 2** — `src/backtest/`: motore di backtest delle strategie
- **Fase 3** — `src/ml/`: modelli predittivi
- **Fase 4** — `src/agent/` + `dashboard/`: agente AI (Claude) e dashboard Streamlit

Le funzioni di `src/analysis/metrics.py` sono già pensate per essere chiamate direttamente come *tool* dell'agente della Fase 4.